In [3]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

In [4]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test_x.csv")
sample = pd.read_csv("../data/sample_submission.csv")

In [5]:
target = "bilissel_performans_skoru"

X = train.drop(target, axis=1)
y = train[target]

In [6]:
cat_features = X.select_dtypes(include=["object"]).columns.tolist()

for col in cat_features:
    X[col] = X[col].fillna("missing")
    test[col] = test[col].fillna("missing")

In [7]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y)):
    print(f"\n--- Fold {fold+1} ---")
    
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_valid, y_valid = X.iloc[valid_idx], y.iloc[valid_idx]
    
    model = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        early_stopping_rounds=100,
        verbose=200
    )
    
    model.fit(
        X_train, y_train,
        eval_set=(X_valid, y_valid),
        cat_features=cat_features,
        use_best_model=True
    )
    
    # OOF Predictions
    oof_preds[valid_idx] = model.predict(X_valid)
    
    # Test Predictions
    test_preds += model.predict(test) / kf.n_splits


--- Fold 1 ---
0:	learn: 2.1743190	test: 2.1846648	best: 2.1846648 (0)	total: 65ms	remaining: 2m 9s
200:	learn: 1.2107060	test: 1.2316761	best: 1.2316761 (200)	total: 1.34s	remaining: 12s
400:	learn: 1.1866587	test: 1.2236367	best: 1.2236273 (396)	total: 2.7s	remaining: 10.8s
600:	learn: 1.1705843	test: 1.2226610	best: 1.2226610 (600)	total: 3.95s	remaining: 9.2s
800:	learn: 1.1566635	test: 1.2224001	best: 1.2223234 (796)	total: 5.19s	remaining: 7.77s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.222319833
bestIteration = 805

Shrink model to first 806 iterations.

--- Fold 2 ---
0:	learn: 2.1776378	test: 2.1691961	best: 2.1691961 (0)	total: 7.37ms	remaining: 14.7s
200:	learn: 1.2115022	test: 1.2263847	best: 1.2263847 (200)	total: 1.24s	remaining: 11.1s
400:	learn: 1.1878120	test: 1.2181778	best: 1.2181778 (400)	total: 2.44s	remaining: 9.75s
600:	learn: 1.1709513	test: 1.2173723	best: 1.2173459 (594)	total: 3.74s	remaining: 8.71s
Stopped by overfitting detector 

In [8]:
oof_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print(f"\nOverall OOF RMSE: {oof_rmse:.4f}")


Overall OOF RMSE: 1.2173


In [9]:
test_preds = test_preds.clip(0, 10)

submission = pd.DataFrame({
    "id": test["id"],
    "bilissel_performans_skoru": test_preds
})

submission.to_csv("../submissions/sub_cv.csv", index=False)
submission.head()

,id,bilissel_performans_skoru
0,1,6.023616
1,2,6.724807
2,3,3.072954
3,4,7.158736
4,5,3.732015
